# LLM Integration Helper - AWS Bedrock, OpenAI, and Ollama


## 1. Required Libraries and Setup

In [1]:
import boto3
import json
import requests
import openai
from typing import Dict, Optional
import os

AWS_ACCESS_KEY_ID = 'AWL'
AWS_SECRET_ACCESS_KEY = 'mR'


print("Libraries imported successfully")

Libraries imported successfully


## 2. AWS Bedrock Setup and Implementation

AWS Bedrock provides access to foundation models like Llama, Claude, and others through a managed service.

In [2]:
# AWS Bedrock Configuration
AWS_REGION = 'us-east-1'
BEDROCK_MODEL_ID = 'us.meta.llama3-3-70b-instruct-v1:0'

def setup_bedrock_client():
    """Initialize AWS Bedrock client"""
    try:
        client = boto3.client(
            'bedrock-runtime',
            region_name=AWS_REGION,
            aws_access_key_id=AWS_ACCESS_KEY_ID,
            aws_secret_access_key=AWS_SECRET_ACCESS_KEY
        )
        print("Bedrock client initialized successfully")
        return client
    except Exception as e:
        print(f"Error initializing Bedrock client: {e}")
        return None

bedrock_client = setup_bedrock_client()

Bedrock client initialized successfully


In [3]:
def call_bedrock_llm(system_prompt: str, user_prompt: str, max_tokens: int = 2000, temperature: float = 0.1) -> str:
    """Make a call to AWS Bedrock Llama model"""
    
    # Construct the prompt in Llama format
    full_prompt = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n{system_prompt}<|eot_id|><|start_header_id|>user<|end_header_id|>\n{user_prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>"
    
    # Prepare the request body
    request_body = {
        "prompt": full_prompt,
        "max_gen_len": max_tokens,
        "temperature": temperature,
        "top_p": 0.9
    }
    
    try:
        response = bedrock_client.invoke_model(
            modelId=BEDROCK_MODEL_ID,
            contentType='application/json',
            accept='application/json',
            body=json.dumps(request_body)
        )
        
        # Parse the response
        response_body = json.loads(response['body'].read())
        return response_body['generation'].strip()
        
    except Exception as e:
        print(f"Error calling Bedrock: {e}")
        return f"Error: {str(e)}"

print("Bedrock LLM function defined")

Bedrock LLM function defined


In [4]:
# Test Bedrock LLM
if bedrock_client:
    system_prompt = "You are a helpful assistant that classifies emails."
    user_prompt = "Classify this email as urgent, important, or promotional: 'Meeting tomorrow at 2 PM with the CEO.'"
    
    result = call_bedrock_llm(system_prompt, user_prompt)
    print("Bedrock Response:")
    print(result)
else:
    print("Bedrock client not available - skipping test")

Bedrock Response:
I would classify this email as "Urgent". The mention of a meeting with a high-level executive (the CEO) scheduled for the very next day implies a sense of time sensitivity and requires the recipient's prompt attention to prepare and attend the meeting.


## 3. OpenAI API Setup and Implementation

OpenAI provides access to GPT models through their API service.

In [5]:
# OpenAI Configuration

OPENAI_API_KEY = "sk-proj-"

# Set environment variables
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

OPENAI_MODEL = 'gpt-4o-mini'

def setup_openai_client():
    """Initialize OpenAI client"""
    try:
        if not OPENAI_API_KEY:
            print("OpenAI API key not found in environment")
            return None
            
        client = openai.OpenAI(api_key=OPENAI_API_KEY)
        print("OpenAI client initialized successfully")
        return client
    except Exception as e:
        print(f"Error initializing OpenAI client: {e}")
        return None

openai_client = setup_openai_client()

OpenAI client initialized successfully


In [6]:
def call_openai_llm(system_prompt: str, user_prompt: str, max_tokens: int = 2000, temperature: float = 0.1) -> str:
    """Make a call to OpenAI GPT model"""
    
    try:
        response = openai_client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=0.9
        )
        
        return response.choices[0].message.content.strip()
        
    except Exception as e:
        print(f"Error calling OpenAI: {e}")
        return f"Error: {str(e)}"

print("OpenAI LLM function defined")

OpenAI LLM function defined


In [7]:
# Test OpenAI LLM
if openai_client:
    system_prompt = "You are a helpful assistant that classifies emails."
    user_prompt = "Classify this email as urgent, important, or promotional: 'Meeting tomorrow at 2 PM with the CEO.'"
    
    result = call_openai_llm(system_prompt, user_prompt)
    print("OpenAI Response:")
    print(result)
else:
    print("OpenAI client not available - skipping test")

OpenAI Response:
This email can be classified as **urgent**.


## 4. Ollama Setup and Implementation

Ollama allows you to run large language models locally on your machine.

In [17]:
# Ollama Configuration
OLLAMA_BASE_URL = 'http://localhost:11434'
OLLAMA_MODEL = 'llama3.1:latest'

def check_ollama_status():
    """Check if Ollama server is running"""
    try:
        response = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
        if response.status_code == 200:
            models = response.json()
            print(f"Ollama server is running. Available models: {len(models.get('models', []))}")
            return True
        else:
            print(f"Ollama server responded with status {response.status_code}")
            return False
    except Exception as e:
        print(f"Ollama server not accessible: {e}")
        return False

ollama_available = check_ollama_status()

Ollama server is running. Available models: 1


In [18]:
def call_ollama_llm(system_prompt: str, user_prompt: str, max_tokens: int = 2000, temperature: float = 0.1) -> str:
    """Make a call to local Ollama model"""
    
    # Combine system and user prompts for Ollama
    combined_prompt = f"System: {system_prompt}\n\nUser: {user_prompt}\n\nAssistant:"
    
    request_data = {
        "model": OLLAMA_MODEL,
        "prompt": combined_prompt,
        "stream": False,
        "options": {
            "temperature": temperature,
            "top_p": 0.9,
            "num_predict": max_tokens
        }
    }
    
    try:
        response = requests.post(
            f"{OLLAMA_BASE_URL}/api/generate",
            json=request_data,
            timeout=60
        )
        
        if response.status_code == 200:
            result = response.json()
            return result.get('response', '').strip()
        else:
            return f"Error: HTTP {response.status_code}"
            
    except Exception as e:
        print(f"Error calling Ollama: {e}")
        return f"Error: {str(e)}"

print("Ollama LLM function defined")

Ollama LLM function defined


In [19]:
# Test Ollama LLM
if ollama_available:
    system_prompt = "You are a helpful assistant that classifies emails."
    user_prompt = "Classify this email as urgent, important, or promotional: 'Meeting tomorrow at 2 PM with the CEO.'"
    
    result = call_ollama_llm(system_prompt, user_prompt)
    print("Ollama Response:")
    print(result)
else:
    print("Ollama not available - skipping test")

Ollama Response:
I would classify this email as "Urgent" because it requires immediate attention and action from the recipient (attending a meeting). The fact that it's scheduled for the next day also adds to its sense of urgency.
